In [60]:
import pandas as pd

base_df = pd.read_csv('data/movies_metadata.csv', low_memory=False)

base_df = base_df.drop_duplicates(subset=['title', 'year'], ignore_index=True)

display(base_df)

,title,year,certificate,duration,genre,rating,description,stars,votes
0,Cobra Kai,(2018– ),TV-14,30 min,"Action, Comedy, Drama",8.5,Decades after their 1984 All Valley Karate Tou...,"['Ralph Macchio, ', 'William Zabka, ', 'Courtn...","177,031"
1,The Crown,(2016– ),TV-MA,58 min,"Biography, Drama, History",8.7,Follows the political rivalries and romance of...,"['Claire Foy, ', 'Olivia Colman, ', 'Imelda St...","199,885"
2,Better Call Saul,(2015–2022),TV-MA,46 min,"Crime, Drama",8.9,The trials and tribulations of criminal lawyer...,"['Bob Odenkirk, ', 'Rhea Seehorn, ', 'Jonathan...","501,384"
3,Devil in Ohio,(2022),TV-MA,356 min,"Drama, Horror, Mystery",5.9,When a psychiatrist shelters a mysterious cult...,"['Emily Deschanel, ', 'Sam Jaeger, ', 'Gerardo...","9,773"
4,Cyberpunk: Edgerunners,(2022– ),TV-MA,24 min,"Animation, Action, Adventure",8.6,A Street Kid trying to survive in a technology...,"['Zach Aguilar, ', 'Kenichiro Ohashi, ', 'Emi ...","15,413"
...,...,...,...,...,...,...,...,...,...
8035,Titans,(2018– ),TV-MA,50 min,"Action, Adventure, Crime",5.6,The Titans battle Deathstroke and more.,"['Carol Banker', '| ', ' Stars:', 'Brenton ...","3,492"
8036,The Drew Barrymore Show,(2020– ),TV-PG,44 min,Talk-Show,6.0,In the inaugural episode of The Drew Barrymore...,"['Adam Heydt', '| ', ' Stars:', 'Drew Barry...",19
8037,Hollywood Insider,(2018– ),NaN,NaN,Talk-Show,NaN,Behind the scenes of The Irishman.,"['Bobby Cannavale, ', 'Robert De Niro, ', 'Al ...",NaN
8038,Chocolate,(2019–2020),NaN,NaN,"Drama, Romance",8.1,"Just before his life is turned upside down, Le...","['Lee Hyeong-min', '| ', ' Stars:', 'Ha Ji-...",22


In [61]:
df = base_df[['title', 'votes', 'rating']].copy()
display(df)

,title,votes,rating
0,Cobra Kai,"177,031",8.5
1,The Crown,"199,885",8.7
2,Better Call Saul,"501,384",8.9
3,Devil in Ohio,"9,773",5.9
4,Cyberpunk: Edgerunners,"15,413",8.6
...,...,...,...
8035,Titans,"3,492",5.6
8036,The Drew Barrymore Show,19,6.0
8037,Hollywood Insider,NaN,NaN
8038,Chocolate,22,8.1


In [62]:
df['votes'] = pd.to_numeric(df['votes'], errors='coerce')
df['rating'] = pd.to_numeric(df['rating'], errors='coerce') 

df = df.dropna(subset=['votes', 'rating'])

C = df['rating'].mean()
m = df['votes'].quantile(0.90)

print(f"Середня оцінка (C): {C}")
print(f"Мінімальна кількість голосів (m): {m}")

test_df = df.loc[df['votes'] >= m].copy()
display(test_df)

Середня оцінка (C): 6.285610200364299
Мінімальна кількість голосів (m): 809.0


,title,votes,rating
9,Blonde,935.0,6.2
17,1899,853.0,9.6
204,The Anthrax Attacks,865.0,6.2
491,Superwog,986.0,7.4
503,Lou,880.0,5.9
...,...,...,...
7080,The Untold Tales of Armistead Maupin,809.0,7.6
7131,That's It,892.0,6.8
7217,Craig Ferguson: I'm Here to Help,823.0,7.1
7341,Lucas Brothers: On Drugs,842.0,5.7


In [63]:
v = test_df['votes']
R = test_df['rating']

test_df['score'] = (v / (v + m) * R) + (m / (v + m) * C)

test_df = test_df.sort_values('score', ascending=False)
display(test_df.head(20))

,title,votes,rating,score
17,1899,853.0,9.6,7.986678
2961,Yungnyong-i Nareusya,994.0,8.9,7.726932
7352,#HappyBirthdaySense8,844.0,9.0,7.671542
3578,Lastman,985.0,8.6,7.556331
4611,Daughters of Destiny,935.0,8.6,7.526410
3297,Puffin Rock,932.0,8.6,7.524560
4330,Shinchan,969.0,8.4,7.437941
4598,Like in the Movies,840.0,8.5,7.413620
2380,Ask the StoryBots,904.0,8.4,7.401435
2433,Stove League,918.0,8.3,7.356374


In [64]:
from sklearn.feature_extraction.text import TfidfVectorizer

df = base_df.copy()

df['description'] = df['description'].fillna('')

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['description'])

print("Розмірність матриці:", tfidf_matrix.shape)

Розмірність матриці: (8040, 19678)


In [65]:
display(base_df['description'])
df = base_df.copy()

0       Decades after their 1984 All Valley Karate Tou...
1       Follows the political rivalries and romance of...
2       The trials and tribulations of criminal lawyer...
3       When a psychiatrist shelters a mysterious cult...
4       A Street Kid trying to survive in a technology...
                              ...                        
8035              The Titans battle Deathstroke and more.
8036    In the inaugural episode of The Drew Barrymore...
8037                   Behind the scenes of The Irishman.
8038    Just before his life is turned upside down, Le...
8039    Businessman Jerry Buss bets it all on the lack...
Name: description, Length: 8040, dtype: object

In [66]:
tfidf = TfidfVectorizer(stop_words='english')
df['description'] = df['description'].fillna('')
tfidf_matrix = tfidf.fit_transform(df['description'])

from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

indices = pd.Series(df.index, index=df['title']).drop_duplicates()
display(indices.head())

title
Cobra Kai                 0
The Crown                 1
Better Call Saul          2
Devil in Ohio             3
Cyberpunk: Edgerunners    4
dtype: int64

In [67]:
def get_recommendations(title, df, indices, cosine_sim_matrix):
    if title not in indices:
        return f"Помилка: Фільм '{title}' не знайдено у базі даних."
    
    idx = indices[title]
    
    if isinstance(idx, pd.Series):
        idx = idx.iloc[0]
        
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))
    
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:11]
    
    movie_indices = [i[0] for i in sim_scores]
    
    similarity_values = [i[1] for i in sim_scores]
    
    result_df = pd.DataFrame({
        'title': df.iloc[movie_indices]['title'].values,
        'similarity_score': similarity_values
    })
    
    return result_df

In [68]:
display(get_recommendations('The Crown', df, indices, cosine_sim))

,title,similarity_score
0,Kevin Hart: Don't F**k This Up,0.205746
1,Downton Abbey,0.186587
2,The 80's India,0.183384
3,Greatest Events of WWII in Colour,0.146217
4,The Tudors,0.143176
5,Titletown High,0.139562
6,Sweet Tooth,0.139175
7,Stavisky,0.138551
8,What's Wrong with Secretary Kim,0.138370
9,Guga-ui Seo,0.128161


In [69]:
display(get_recommendations('Cyberpunk: Edgerunners', df, indices, cosine_sim))

,title,similarity_score
0,Wraith,0.153459
1,The Future Of,0.130779
2,On My Block,0.126032
3,Slasher,0.114848
4,Ridley Jones,0.110630
5,Lego Friends: Girls on A Mission,0.110122
6,Tell No One,0.108490
7,The Walking Dead,0.108487
8,Into the Night,0.108014
9,A Dog's Life: The Oscar Lose Story,0.104373
